## Универсальный инференс и генерация сабмита (BERT / TF-IDF+LR / ансамбль)
Скрипт читает тестовый датасет `test_super.csv`, по пути `MODEL_PATH` автоматически определяет тип модели (папка с BERT, sklearn-пайплайн `.joblib` или JSON-конфиг ансамбля), считает вероятности классов с возможным кэшированием прогонов, формирует предсказания меток и сохраняет итоговый файл сабмита `submission.csv` в требуемом формате (`ID`, `label`).


In [ ]:
import os
import json
import gc
import time

import numpy as np
import pandas as pd

from joblib import load

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.functional import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification


TEST_TEXT_PATH = "../data/processed/test_super.csv"

# модель:
# MODEL_PATH = "../models/rurorberta_base2_manual"          # BERT
# MODEL_PATH = "../models/sentiment_lr_optuna.joblib"      # LR
MODEL_PATH = "../models/ensemble_config_cv.json"           # АНСАМБЛЬ (json)

SUBMISSION_PATH = "../data/ТОНАЛЬНОСТЬ/submission.csv"

MAX_LEN_DEFAULT = 192
BATCH_SIZE_DEFAULT = 32

# КЭШИ ДЛЯ ТЕСТОВЫХ ПРОБ
BERT_TEST_CACHE = "../data/cache/bert_probs_test_best.npy"
LR_TEST_CACHE   = "../data/cache/lr_probs_test_best.npy"


def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")


class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len: int):
        self.texts = list(texts)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


def detect_model_type(path: str) -> str:
    if os.path.isdir(path) and os.path.exists(os.path.join(path, "config.json")):
        return "bert_dir"
    if os.path.isfile(path):
        if path.endswith(".joblib"):
            return "joblib"
        if path.endswith(".json"):
            return "ensemble"
    raise ValueError(
        f"Не удалось определить тип модели по MODEL_PATH={path}\n"
        f"Ожидается папка с config.json, .joblib или .json."
    )


# ====== PREDICT: BERT ======

def get_bert_probs_for_texts(
    texts: pd.Series,
    model_dir: str,
    max_len: int = MAX_LEN_DEFAULT,
    batch_size: int = BATCH_SIZE_DEFAULT,
    cache_path: str | None = None,
) -> np.ndarray:
    if cache_path is not None and os.path.exists(cache_path):
        print(f"🔁 Загружаю BERT-пробы из кэша: {cache_path}")
        return np.load(cache_path)

    t0 = time.time()
    device = get_device()
    print("Устройство для BERT:", device)

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    ds = InferenceDataset(texts, tokenizer, max_len=max_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    all_probs = []
    with torch.no_grad():
        for i, batch in enumerate(dl):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            probs = softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())

    all_probs = np.vstack(all_probs)

    del model, tokenizer
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    dt = time.time() - t0
    print(f"⏱ BERT-прогон занял {dt:.1f} c")

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        np.save(cache_path, all_probs)
        print(f"💾 BERT-пробы сохранены в {cache_path}")

    return all_probs


# ====== PREDICT: joblib (TF-IDF + LR) ======

def get_joblib_probs_for_texts(
    texts: pd.Series,
    joblib_path: str,
    cache_path: str | None = None,
) -> np.ndarray:
    if cache_path is not None and os.path.exists(cache_path):
        print(f"🔁 Загружаю LR-пробы из кэша: {cache_path}")
        return np.load(cache_path)

    if not os.path.exists(joblib_path):
        raise FileNotFoundError(f"Не найден {joblib_path}")

    t0 = time.time()
    pipeline = load(joblib_path)
    probs = None

    if hasattr(pipeline, "predict_proba"):
        probs = pipeline.predict_proba(texts.astype(str))
    else:
        preds = pipeline.predict(texts.astype(str))
        classes = np.unique(preds)
        num_classes = len(classes)
        oh = np.zeros((len(preds), num_classes), dtype=float)
        for i, c in enumerate(preds):
            oh[i, int(c)] = 1.0
        probs = oh

    dt = time.time() - t0
    print(f"⏱ TF-IDF+LR-прогон занял {dt:.1f} c")

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        np.save(cache_path, probs)
        print(f"💾 LR-пробы сохранены в {cache_path}")

    return probs


# ====== PREDICT: ENSEMBLE (BERT + joblib по json конфигу) ======

def get_ensemble_probs_from_config(
    texts: pd.Series,
    config_path: str,
    max_len_default: int = MAX_LEN_DEFAULT,
    batch_size_default: int = BATCH_SIZE_DEFAULT,
) -> np.ndarray:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    print("\nКонфиг ансамбля (из json):")
    print(json.dumps(cfg, ensure_ascii=False, indent=2))

    cfg_dir = os.path.dirname(os.path.abspath(config_path))

    bert_model_dir = cfg["bert_model_dir"]
    lr_model_path = cfg["lr_model_path"]
    w_bert = cfg["bert_weight"]
    w_lr = cfg["lr_weight"]
    max_len = cfg.get("max_len", max_len_default)
    batch_size = cfg.get("batch_size", batch_size_default)

    if not os.path.isabs(bert_model_dir):
        bert_model_dir = os.path.join(cfg_dir, bert_model_dir)
    if not os.path.isabs(lr_model_path):
        lr_model_path = os.path.join(cfg_dir, lr_model_path)

    print(f"\nHF-модель: {bert_model_dir}")
    print(f"joblib:    {lr_model_path}")
    print(f"Веса: w_bert={w_bert:.3f}, w_lr={w_lr:.3f}")
    print(f"max_len={max_len}, batch_size={batch_size}")

    bert_probs = get_bert_probs_for_texts(
        texts,
        bert_model_dir,
        max_len=max_len,
        batch_size=batch_size,
        cache_path=BERT_TEST_CACHE,
    )

    lr_probs = get_joblib_probs_for_texts(
        texts,
        lr_model_path,
        cache_path=LR_TEST_CACHE,
    )

    assert bert_probs.shape == lr_probs.shape, \
        f"bert_probs shape {bert_probs.shape} != lr_probs shape {lr_probs.shape}"

    final_probs = w_bert * bert_probs + w_lr * lr_probs
    return final_probs



def main():
    if not os.path.exists(TEST_TEXT_PATH):
        raise FileNotFoundError(f"Не найден {TEST_TEXT_PATH}")
    print(f"Читаем тестовые тексты из {TEST_TEXT_PATH}...")
    texts_df = pd.read_csv(TEST_TEXT_PATH)

    for col in ["ID", "text"]:
        if col not in texts_df.columns:
            raise ValueError(f"В файле с текстами должна быть колонка '{col}'")

    texts_df["ID"] = texts_df["ID"].astype(int)
    texts = texts_df["text"].astype(str)
    print("Размер теста:", len(texts))

    # --- определяем тип модели ---
    mtype = detect_model_type(MODEL_PATH)
    print(f"\nОпределён тип модели: {mtype} (по пути {MODEL_PATH})")

    if mtype == "bert_dir":
        print("\nДелаем предсказания BERT/ruBERT моделью...")
        probs = get_bert_probs_for_texts(
            texts,
            MODEL_PATH,
            max_len=MAX_LEN_DEFAULT,
            batch_size=BATCH_SIZE_DEFAULT,
            cache_path=BERT_TEST_CACHE,
        )
        preds = probs.argmax(axis=1)

    elif mtype == "joblib":
        print("\nДелаем предсказания joblib (sklearn-пайплайном)...")
        probs = get_joblib_probs_for_texts(
            texts,
            MODEL_PATH,
            cache_path=LR_TEST_CACHE,
        )
        preds = probs.argmax(axis=1)

    elif mtype == "ensemble":
        print("\nДелаем предсказания ансамблем по json-конфигу...")
        probs = get_ensemble_probs_from_config(
            texts,
            MODEL_PATH,
            max_len_default=MAX_LEN_DEFAULT,
            batch_size_default=BATCH_SIZE_DEFAULT,
        )
        preds = probs.argmax(axis=1)

    else:
        raise ValueError(f"Неизвестный тип модели: {mtype}")

    submission_df = pd.DataFrame({
        "ID": texts_df["ID"].values,
        "label": preds.astype(int),
    })

    os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
    submission_df.to_csv(SUBMISSION_PATH, index=False)
    print(f"\n✅ Сабмит сохранён в {SUBMISSION_PATH}")
    print("Первые строки:")
    print(submission_df.head())


if __name__ == "__main__":
    main()


Читаем тестовые тексты из ../data/processed/test_super.csv...


The tokenizer you are loading from '/Users/olgashalashova/sentiment_project/models/../models/rurorberta_base2_manual' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Размер теста: 58092

Определён тип модели: ensemble (по пути ../models/ensemble_config_cv.json)

Делаем предсказания ансамблем по json-конфигу...

Конфиг ансамбля (из json):
{
  "bert_model_dir": "../models/rurorberta_base2_manual",
  "lr_model_path": "../models/sentiment_lr_optuna.joblib",
  "bert_weight": 0.35000000000000003,
  "lr_weight": 0.6499999999999999,
  "num_labels": 3,
  "max_len": 192,
  "batch_size": 32,
  "n_splits": 5,
  "cv_f1_macro": 0.90743968911667
}

HF-модель: /Users/olgashalashova/sentiment_project/models/../models/rurorberta_base2_manual
joblib:    /Users/olgashalashova/sentiment_project/models/../models/sentiment_lr_optuna.joblib
Веса: w_bert=0.350, w_lr=0.650
max_len=192, batch_size=32
Устройство для BERT: mps
⏱ BERT-прогон занял 568.0 c
💾 BERT-пробы сохранены в ../data/cache/bert_probs_test_best.npy
⏱ TF-IDF+LR-прогон занял 17.1 c
💾 LR-пробы сохранены в ../data/cache/lr_probs_test_best.npy

✅ Сабмит сохранён в ../data/ТОНАЛЬНОСТЬ/submission.csv
Первые строки: